# 04. 대화형 Agent 완성

---
## 0. 설치 & 경로

In [1]:
!pip install -U duckduckgo-search

  Using cached duckduckgo_search-8.1.1-py3-none-any.whl.metadata (16 kB)
Using cached duckduckgo_search-8.1.1-py3-none-any.whl (18 kB)
   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   -------- ------------------------------- 1.0/4.7 MB 9.9 MB/s eta 0:00:01
   ------------------------ --------------- 2.9/4.7 MB 7.1 MB/s eta 0:00:01
   -------------------------------------- - 4.5/4.7 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 4.7/4.7 MB 6.4 MB/s  0:00:00

   -------------------- ------------------- 1/2 [duckduckgo-search]
   ---------------------------------------- 2/2 [duckduckgo-search]



In [2]:
!pip install openai python-dotenv langchain langchain-openai langchain-core langchain-community langchain-text-splitters langchain-classic faiss-cpu pypdf pytz yfinance requests ddgs tabulate rank_bm25

  Using cached openai-2.44.0-py3-none-any.whl.metadata (34 kB)
  Using cached pypdf-6.14.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
Using cached openai-2.44.0-py3-none-any.whl (1.4 MB)
   ---------------------------------------- 0.0/558.3 kB ? eta -:--:--
   ---------------------------------------- 558.3/558.3 kB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/652.6 kB ? eta -:--:--
   ---------------------------------------- 652.6/652.6 kB 16.7 MB/s  0:00:00
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ---------------------------------------- 874.8/874.8 kB 16.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 21.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------

In [7]:
!pip install langchain-chroma chromadb

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl (23.5 MB)
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ------------------------------- -------- 3.9/5.0 MB 23.1 MB/s eta 0:00:01
   ---------------------------------------- 5.0/5.0 MB 19.9 MB/s  0:00:00
   ---------------------------------------- 0.0/4.6 MB ? eta -:--:--


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

RAG_SAMPLES = os.path.join(os.getcwd(), 'samples', 'pdf_samples')
print('RAG_SAMPLES:', RAG_SAMPLES)
for p in sorted(Path(RAG_SAMPLES).glob('*.pdf')):
    print(' -', p.name)

RAG_SAMPLES: c:\Users\Admin\Desktop\실습폴더\14일차_실습\samples\pdf_samples
 - A survey on large language model based autonomous agents.pdf
 - data2vec.pdf
 - NeurIPS-2023-analyzing-vision-transformers-for-image-classification-in-class-embedding-space-Paper-Conference.pdf
 - Physics-Guided Deep Learning.pdf
 - supervised-contrastive-learning-Paper.pdf
 - 보도자료.pdf
 - 학칙.pdf


---
## Step 1. RAG VectorStore 

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 학칙.pdf
loader = PyPDFLoader(os.path.join(RAG_SAMPLES, '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())

# news.pdf
loader_news = PyPDFLoader(os.path.join(RAG_SAMPLES, '보도자료.pdf'))
news_splits = text_splitter.split_documents(loader_news.load())
all_splits.extend(news_splits)

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large', api_key=OPENAI_API_KEY
)

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

Creating new Chroma store


In [9]:
retriever # 생성한 VectorStore 기반으로 검색을 수행하는 Retriever 객체

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000270DA921D30>, search_kwargs={'k': 3})

---
## Step 1.1 검색 품질 점검

`석사학위과정 수업연한`처럼 **별표(부록)에 같은 단어가 반복**되는 PDF에서는, 기본 유사도 검색이 엉뚱한 청크 3개만 가져올 수 있습니다.  
아래에서 검색 결과와 **실제 정답 조항**을 비교해 봅니다.

In [10]:
demo_query = '석사학위과정 수업연한'

retriever_baseline = vectorstore.as_retriever(search_kwargs={'k': 3})
baseline_hits = retriever_baseline.invoke(demo_query)

print(f'질문: {demo_query}\n')
for i, doc in enumerate(baseline_hits, 1):
    has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
    print(f'[{i}] 수업연한 정답 포함: {has_answer}')
    print(doc.page_content[:350].replace('\n', ' '))
    print('-' * 60)

질문: 석사학위과정 수업연한

[1] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language
------------------------------------------------------------
[2] 수업연한 정답 포함: False
협동과정 비교문학협동과정문학석사M.A. in Comparative LiteraturePANTONE 5315C인지과학협동과정인지과학석사M.S. in Cognitive SciencePANTONE 5315C문학석사M.A. in Cognitive SciencePANTONE 5315C공학석사M.S. in Cognitive Science and EngineeringPANTONE 172C경영학석사M.A. in Cognitive Science and ManagementPANTONE 477C이학석사M.S. in Cognitive SciencePANTONE 136C한국학협동과정문학석사M.A. in Korean StudiesPANTONE 
------------------------------------------------------------
[3] 수업연한 정답 포함: False
Ⅰ. 대학원 학칙  /  29 [별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292

In [11]:
# 학칙.pdf 본문에 실제로 있는 조항 (검색이 찾아야 할 정답)
answer_chunks = [
    s for s in all_splits
    if '수업연한' in s.page_content and '2년(4학기)' in s.page_content
]

print(f'정답 청크 수: {len(answer_chunks)}')
print('=' * 60)
print(answer_chunks[0].page_content[:700])

정답 청크 수: 1
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학연한에 산입하지 아니한다. 1. 석사학위과정: 4년(8학기)2. 박사학위과정: 7년(14학기)3. 통합과정: 8년(16학기)② 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자로서 절차에 따라 학적에 구분표시 되어 있는 학생에 대하여는 재학연한을 제한하지 아니한다.제2조의9(융합전공) ① 대학원 내 두 개 이상의 학과가 융합적 교과과정(융합전공)을 공동 운영할 수 있다.② 융합전공의 운영에 관한 사항은 별도로 정한다.제2조의10(학생설계전공) ① 새로운 학문수요의 충족과 자기 주도적 학습능력의 향상을 위하여 학생설계전공을 둘 수 있다.② 학생설계전공의 설치 및 이수요건은 별도로 정한다.제3조(입학정원) ① 대학원에 두는 학위과정별 입학정원([별표 1])은 대학원운영위원회에서 대학별로 정한다. 단, 통합과정 입학정원은 박사학위과정


---
## Step 1.2 BM25 키워드 검색 + 하이브리드

**BM25**는 단어(키워드) 빈도 기반 검색입니다. `수업연한`처럼 **문서에 정확히 등장하는 용어**를 찾을 때 벡터 검색보다 유리한 경우가 많습니다.

| 방식 | 장점 | 약점 |
|------|------|------|
| 벡터 검색 | 의미가 비슷한 문장도 찾음 | 별표처럼 같은 단어가 반복되면 엉뚱한 청크가 올라옴 |
| BM25 | 키워드가 들어 있는 청크를 직접 찾음 | 한글 PDF는 띄어쓰기가 적어 **토크나이저 설정**이 필요 |

실습에서는 **BM25 + 벡터**를 `EnsembleRetriever`로 합쳐 하이브리드 검색을 씁니다.

In [12]:
import re
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.tools import create_retriever_tool


def ko_preprocess(text):
    """한글 학칙 PDF용 — 띄어쓰기 대신 한글 어절 단위로 토큰 분리"""
    return re.findall(r'[가-힣]+|\d+', text)


def show_hits(label, hits, preview=3):
    print(f'\n=== {label} ===')
    for i, doc in enumerate(hits[:preview], 1):
        has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
        print(f'[{i}] 수업연한 정답 포함: {has_answer}')
        print(doc.page_content[:280].replace('\n', ' '))
        print('-' * 50)

In [13]:
# BM25 키워드 검색
bm25_retriever = BM25Retriever.from_documents(all_splits, preprocess_func=ko_preprocess)
bm25_retriever.k = 3
bm25_hits = bm25_retriever.invoke(demo_query)
show_hits('BM25 키워드 검색 (k=3)', bm25_hits)


=== BM25 키워드 검색 (k=3) ===
[1] 수업연한 정답 포함: True
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학
--------------------------------------------------
[2] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German
--------------------------------------------------
[3] 수업연한 정답 포함: False
10 제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다음 각 

In [14]:
# 하이브리드: BM25 + 벡터
vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)
hybrid_hits = hybrid_retriever.invoke(demo_query)
show_hits('하이브리드 BM25 + 벡터 (k=3)', hybrid_hits)

found = any(
    '수업연한' in d.page_content and '2년(4학기)' in d.page_content
    for d in hybrid_hits
)
print('\n수업연한 정답 청크 포함 여부:', found)


=== 하이브리드 BM25 + 벡터 (k=3) ===
[1] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German
--------------------------------------------------
[2] 수업연한 정답 포함: True
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학
--------------------------------------------------
[3] 수업연한 정답 포함: False
협동과정 비교문학협동과정문학석사M.A. in Comparative LiteraturePANTONE 5315C인지과학협동과정인지과학석사M.S. in Cognitive SciencePANTONE 5315C문학석사M.A. in Cognitive SciencePANTONE 5315C공학석사M.S. in Cognitive Science and EngineeringPANTONE 172C경영학석사M.A. in Cognitive Scienc

---
## Step 2. Retriever → Tool (5.3 마지막 섹션)

`langchain_core.tools.create_retriever_tool` — 5.3과 동일합니다.

> **Step 1.2**에서 하이브리드 retriever로 `search_pdf_tool`을 이미 만들었다면, 이 셀은 생략해도 됩니다.

In [ ]:
from langchain_core.tools import create_retriever_tool

search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)
#description이 길수록 이 설명을 읽는데 토큰 소모량 증가

---
## Step 3. 유틸 Tool (5.2 복습)

시각 · 주가 — `@tool` 로 등록합니다.

In [16]:
from datetime import datetime
import pytz
import yfinance as yf
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'


In [17]:
print(get_current_time.invoke({'timezone': 'Asia/Seoul', 'location': '서울'}))

Asia/Seoul (서울) 현재시각 2026-07-09 14:23:33


---
## Step 4. DuckDuckGo 웹 검색

최신 뉴스·시사는 PDF RAG가 아니라 **웹 검색** Tool을 씁니다.  
Agent가 질문마다 `search_pdf_documents` vs `duckduckgo_results_json` 중 선택합니다.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

web_search = DuckDuckGoSearchResults(num_results=3)
print(web_search.name, '-', web_search.description[:80], '...')
#llm이 요약해준 검색 결과

duckduckgo_results_json - A wrapper around Duck Duck Go Search. Useful for when you need to answer questio ...


In [19]:
web_search.invoke("26년 월드컵 한국 vs 멕시코 점수는?")

'snippet: 3 weeks ago - 목요일 저녁(현지 시간) 멕시코 할리스코주 과달라하라에서 열린 국제축구연맹(FIFA) 월드컵 2026 대회 A조 두 번째 경기에서 한국이 멕시코에 0-1로 패하며 1승 1패를 기록했습니다., title: 축구: 대한민국 0-1 멕시코 - FIFA 월드컵 2026, link: https://www.olympics.com/ko/news/football-korea-mexico-fifa-world-cup-2026, snippet: 3 weeks ago - 6월 19일 오전 10시, 2026 북중미 월드컵 A조 2차전 한국 대 멕시코 경기가 멕시코 과달라하라 에스타디오 아크론에서 열렸습니다. 1차전에서 체코를 꺾고 승점 3점을 챙긴 한국이 공동개최국 ..., title: 2026 월드컵 한국 멕시코 결과와 하이라이트 다시보기 총정리, link: https://moneyroan.com/korea-vs-mexico-worldcup-2026-result/, snippet: 3 weeks ago - 18일 목요일(현지시간) ... 멕시코의 홈에서 멕시코를 상대로 2연승을 노렸지만, 0-1로 패하며 1승 1패를 기록했습니다...., title: 축구: 한국·멕시코·남아공 조별 예선 2차전 라이브 스코어·경기 결과 - FIFA 월드컵 2026, link: https://www.olympics.com/ko/news/fifa-world-cup-2026-live-scores-18-june-results'

---
## Step 5. AgentExecutor — Tool 통합

5.2 수동 tool 루프 대신 **AgentExecutor**가 tool_calls 루프를 자동 처리합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)
### verbose 진행중인 결과를 보여주는 함수

['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json']


In [21]:
# PDF → search_pdf_documents
r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사학위과정 수업연한'}`


34
[별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드
문과대학
국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language and LiteraturePANTONE 5315C노어노문학과문학석사M.A. in Russian Language and LiteraturePANTONE 5315C사학과문학석사M.A. in HistoryPANTONE 5315C철학과문학석사M.A. in PhilosophyPANTONE 5315C문헌정보학과문헌정보학석사M.A. in Library and Information SciencePANTONE 5315C기록관리학석사M.A. in Record Management and ArchivesPANTONE 5315C심리학과문학석사M.A. in PsychologyPANTONE 5315C심리학석사M.S. in PsychologyPANTONE 5315C인지과학석사M.S. in Cognitive SciencePANTONE 5315C
협동과정

협동과정
비교문학협동과정문학석사M.A. in Comparative LiteraturePANTONE 5315C인지과학협동과정인지과학석사M.S. in Cognitive SciencePANTONE 5315C문학석사M.A. in Cognit

In [22]:
search_pdf_tool = create_retriever_tool(
    hybrid_retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)
# PDF → search_pdf_documents

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사학위과정 수업연한'}`


34
[별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드
문과대학
국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language and LiteraturePANTONE 5315C노어노문학과문학석사M.A. in Russian Language and LiteraturePANTONE 5315C사학과문학석사M.A. in HistoryPANTONE 5315C철학과문학석사M.A. in PhilosophyPANTONE 5315C문헌정보학과문헌정보학석사M.A. in Library and Information SciencePANTONE 5315C기록관리학석사M.A. in Record Management and ArchivesPANTONE 5315C심리학과문학석사M.A. in PsychologyPANTONE 5315C심리학석사M.S. in PsychologyPANTONE 5315C인지과학석사M.S. in Cognitive SciencePANTONE 5315C
협동과정

③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 

In [23]:
# 시각 → get_current_time
r = executor.invoke({'input': '부산 지금 몇시야?', 'chat_history': []})
print('시각:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{'timezone': 'Asia/Seoul', 'location': '부산'}`


Asia/Seoul (부산) 현재시각 2026-07-09 14:29:49부산의 현재 시각은 2026년 7월 9일 14시 29분 49초입니다.

> Finished chain.
시각: 부산의 현재 시각은 2026년 7월 9일 14시 29분 49초입니다.


---
## Step 6. 멀티턴 — `chat_history` (5.1 + Agent)

In [24]:
print(executor.invoke({'input': '학칙에서 석사 수업연한 알려줘', 'chat_history': []})['output'])
print(executor.invoke({'input': '방금 말한 내용 한 줄로 다시', 'chat_history': []})['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사 수업연한'}`


Ⅰ. 대학원 학칙  /  29
[별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292)국제12271합계1,4501,1001) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 석사 정원 8명 증원2) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 박사 정원 3명 증원□ 계약학과 학위과정 [정원 외]1. 신촌캠퍼스구분석사 정원(명)박사 정원(명)재교육형10028채용조건형11733합계21761

Ⅰ. 대학원 학칙  /  15

10
제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다음 각 항에서 정한 등록을 하여야 하며, 등록은 정규등록과 연구등록으로 구분하고, 등록 시 등록금을 납부하여야 한다.② 정규등록은 각 학위과정 수업연한 내 학기의 등록을 말하며, 연구등록은 각 학위과정의 수업연한을 초과한 학기의 등록을 말한다.③ <삭제>④ 각 학위과정의 학생 중 학위수여 자격 요건을 충족한 학생은 정규등록을 한 학기 단축할 수 있다. 단, 통합과정의 학생 및 석사학위 논문 대체실적을 통해 학위를 취득하고자 하는 자는 제외한다.⑤ <삭제> 제9조(재학생의 권한) 재학생은 학점을 이수할 수 있고, 학위논문 제출 자격시험 등에 응시할 수 있으며, 학위논문 연구지도를 받을 수 있다. 제10조(등록금의 납부 및 반환) ① <삭제>② 등록금 납부

In [26]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('학칙에서 석사 수업연한 알려줘'))
print('2:', session.ask('방금 답한 내용 한 줄로 다시'))



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사 수업연한'}`


Ⅰ. 대학원 학칙  /  29
[별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292)국제12271합계1,4501,1001) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 석사 정원 8명 증원2) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 박사 정원 3명 증원□ 계약학과 학위과정 [정원 외]1. 신촌캠퍼스구분석사 정원(명)박사 정원(명)재교육형10028채용조건형11733합계21761

Ⅰ. 대학원 학칙  /  15

10
제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다음 각 항에서 정한 등록을 하여야 하며, 등록은 정규등록과 연구등록으로 구분하고, 등록 시 등록금을 납부하여야 한다.② 정규등록은 각 학위과정 수업연한 내 학기의 등록을 말하며, 연구등록은 각 학위과정의 수업연한을 초과한 학기의 등록을 말한다.③ <삭제>④ 각 학위과정의 학생 중 학위수여 자격 요건을 충족한 학생은 정규등록을 한 학기 단축할 수 있다. 단, 통합과정의 학생 및 석사학위 논문 대체실적을 통해 학위를 취득하고자 하는 자는 제외한다.⑤ <삭제> 제9조(재학생의 권한) 재학생은 학점을 이수할 수 있고, 학위논문 제출 자격시험 등에 응시할 수 있으며, 학위논문 연구지도를 받을 수 있다. 제10조(등록금의 납부 및 반환) ① <삭제>② 등록금 납부

---
## Step 7. `input()` 대화 루프

In [28]:
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))

종료: exit


> Entering new AgentExecutor chain...

Invoking: `get_yf_stock_history` with `{'ticker': '000660.KS', 'period': '1mo'}`


                                Open       High        Low      Close   Volume  Dividends  Stock Splits
Date                                                                                                   
2026-07-07 00:00:00+09:00  2289000.0  2343000.0  2080000.0  2201000.0  6351195        0.0           0.0
2026-07-08 00:00:00+09:00  2118000.0  2329000.0  2076000.0  2076000.0  6975631        0.0           0.0
2026-07-09 00:00:00+09:00  2228000.0  2270000.0  2091000.0  2148000.0  4133346        0.0           0.0
Invoking: `get_yf_stock_history` with `{'ticker': '005930.KS', 'period': '1mo'}`


                               Open      High       Low     Close    Volume  Dividends  Stock Splits
Date                                                                                                
2026-07-07 00:00:00+09:00  307000.0  310000.0  286000.0  29600

---
## Step 8. 나만의 Tool 만들기

`@tool`로 함수를 등록하고 `agent_tools`에 넣으면 Agent가 사용할 수 있습니다.

1. 함수 정의 (`@tool` + docstring)
2. `.invoke()`로 단독 테스트
3. `agent_tools`에 추가 → prompt 보강 → `executor` 재생성
4. Agent 질문으로 Tool 호출 확인

### 8.1 예시 — `celsius_to_fahrenheit`

섭씨 → 화씨 변환 Tool을 만들고 Agent에 연결합니다.

In [29]:
from langchain_core.tools import tool

@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """섭씨 온도를 화씨로 변환합니다."""
    fahrenheit = celsius * 9 / 5 + 32
    return f'{celsius}°C = {fahrenheit:.1f}°F'

# 1) Tool 단독 실행
print(celsius_to_fahrenheit.name)
print(celsius_to_fahrenheit.invoke({'celsius': 25}))

celsius_to_fahrenheit
25.0°C = 77.0°F


In [30]:
# 2) agent_tools에 추가 후 Agent 재생성
agent_tools = agent_tools + [celsius_to_fahrenheit]

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time
- 섭씨→화씨 온도 변환 → celsius_to_fahrenheit

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

print('등록된 Tool:', [t.name for t in agent_tools])

등록된 Tool: ['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json', 'celsius_to_fahrenheit']


In [31]:
# 3) Agent가 새 Tool을 호출하는지 확인
r = executor.invoke({'input': '섭씨 36.5도는 화씨로 몇 도야?', 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `celsius_to_fahrenheit` with `{'celsius': 36.5}`


36.5°C = 97.7°F섭씨 36.5도는 화씨로 97.7도입니다.

> Finished chain.
섭씨 36.5도는 화씨로 97.7도입니다.


### 8.2 실습 — 나만의 Tool

원하는 기능의 Tool을 직접 만들고, Agent가 호출하는지 확인합니다.

- 예: 단위 변환, 문자열 처리, 간단한 계산, 로컬 파일 읽기 등
- docstring에 **언제 쓸지**를 적어야 Agent가 선택합니다.

In [34]:
import re
import statistics

# 1) @tool 함수 정의
@tool
def calculate_basic_statistics(numbers: str) -> str:
    """숫자 목록의 기본 통계량(개수, 합, 평균, 중앙값, 최솟값, 최댓값, 표준편차)을 계산합니다.
    띄어쓰기, 쉼표, 줄바꿈 등으로 구분된 숫자 문자열을 입력하세요.
    예: "10 20 30 40" 또는 "1.5, 2.3, 4.7"
    """
    tokens = re.findall(r'-?\d+(?:\.\d+)?', numbers)
    if not tokens:
        return '유효한 숫자를 찾을 수 없습니다. 띄어쓰기·쉼표로 구분해 입력하세요.'

    values = [float(t) for t in tokens]
    n = len(values)
    mean = statistics.mean(values)
    median = statistics.median(values)
    stdev = statistics.stdev(values) if n >= 2 else 0.0

    return (
        f'입력값: {values}\n'
        f'개수: {n}\n'
        f'합: {sum(values):.4g}\n'
        f'평균: {mean:.4g}\n'
        f'중앙값: {median:.4g}\n'
        f'최솟값: {min(values):.4g}\n'
        f'최댓값: {max(values):.4g}\n'
        f'표준편차: {stdev:.4g}'
    )

# 2) 단독 실행 테스트
print(calculate_basic_statistics.name)
print(calculate_basic_statistics.invoke({'numbers': '10 20 30 40 50'}))
print('---')
print(calculate_basic_statistics.invoke({'numbers': '1.5, 2.3, 4.7, 8.1'}))

# 3) agent_tools에 추가 + prompt에 한 줄 안내 + executor 재생성
agent_tools = agent_tools + [calculate_basic_statistics]

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time
- 섭씨→화씨 온도 변환 → celsius_to_fahrenheit
- 숫자 목록의 기본 통계량(평균, 중앙값, 표준편차 등) → calculate_basic_statistics

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

print('등록된 Tool:', [t.name for t in agent_tools])

# 4) Agent 질문으로 Tool 호출 확인
r = executor.invoke({'input': '10 20 30 40 50 이 숫자들의 평균이랑 표준편차 알려줘', 'chat_history': []})
print(r['output'])

calculate_basic_statistics
입력값: [10.0, 20.0, 30.0, 40.0, 50.0]
개수: 5
합: 150
평균: 30
중앙값: 30
최솟값: 10
최댓값: 50
표준편차: 15.81
---
입력값: [1.5, 2.3, 4.7, 8.1]
개수: 4
합: 16.6
평균: 4.15
중앙값: 3.5
최솟값: 1.5
최댓값: 8.1
표준편차: 2.964
등록된 Tool: ['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json', 'celsius_to_fahrenheit', 'calculate_basic_statistics']


> Entering new AgentExecutor chain...

Invoking: `calculate_basic_statistics` with `{'numbers': '10 20 30 40 50'}`


입력값: [10.0, 20.0, 30.0, 40.0, 50.0]
개수: 5
합: 150
평균: 30
중앙값: 30
최솟값: 10
최댓값: 50
표준편차: 15.81주어진 숫자 10, 20, 30, 40, 50의 평균은 30이고, 표준편차는 약 15.81입니다.

> Finished chain.
주어진 숫자 10, 20, 30, 40, 50의 평균은 30이고, 표준편차는 약 15.81입니다.


In [36]:
# 3) Agent가 새 Tool을 호출하는지 확인
r = executor.invoke({'input': '1.7 39 29, 29.34 의 표준 편차는 얼마야?', 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `calculate_basic_statistics` with `{'numbers': '1.7 39 29 29.34'}`


입력값: [1.7, 39.0, 29.0, 29.34]
개수: 4
합: 99.04
평균: 24.76
중앙값: 29.17
최솟값: 1.7
최댓값: 39
표준편차: 16.06주어진 숫자 목록의 표준 편차는 약 16.06입니다.

> Finished chain.
주어진 숫자 목록의 표준 편차는 약 16.06입니다.
